# Fix FITS calibration data
This notebook reprocesses all of the original FITS files to make two fixes:

0. Remove the first file
1. Clip the first row from each file (instead of the last)
2. Adjust the exposure times for each 

In [13]:
import pathlib
import astropy.io.fits
import numpy as np
import tqdm

In [2]:
root_dir = pathlib.Path('../data/moxsi_gsfc_calibration_images')
original_dir = root_dir / 'original'
reprocessed_dir = root_dir / 'reprocessed'

In [3]:
original_files = sorted(original_dir.glob('csie_image_*.bin.fits'))
original_files = original_files[1:]

In [15]:
for i, ofile in tqdm.tqdm(enumerate(original_files)):
    with astropy.io.fits.open(ofile, memmap=False) as hdul:
        # Clip first row
        hdul[0].data = hdul[0].data[1:,:]
        # Fix exposure time
        exp_time = hdul[0].header['EXPTIME']
        if i < 414 and exp_time == 1250:
            exp_time_fixed = 5000
        elif i == 614:
            exp_time_fixed = 5000
        elif i >= 414 and exp_time == 2500:
            exp_time_fixed = 1250
        elif i >= 414 and exp_time == 5000:
            exp_time_fixed = 2500
        else:
            exp_time_fixed = exp_time
        hdul[0].header['EXPTIME'] = exp_time_fixed
        # Write out new file
        reprocessed_dir.mkdir(parents=True, exist_ok=True)
        hdul.writeto(reprocessed_dir / ofile.name)

4598it [00:36, 125.26it/s]


In [4]:
original_files[0].name

'csie_image_20250930_182227_067127_frame53598.bin.fits'

In [11]:
with astropy.io.fits.open(reprocessed_dir / ofile.name) as hdul:
    print(hdul[0].data.shape)
    print(hdul[0].header['EXPTIME'])

(2000, 1504)
5000
